# Distributed Lightweight Model Runner

This notebook provides a quick way to run the distributed lightweight workflow wrapper:
- `run_east_river_distributed_workflow_light.py`
- Presets: `minimal`, `light`, `full`
- Optional model run and local MPI SUMMA flags

In [ ]:
from pathlib import Path
import subprocess
import shlex

repo_root = Path('/home/dlhogan/projects/forked-repos/CONFLUENCE-uwmtnhydro').resolve()
runner = repo_root / 'ess-project/modeling/04-distributed/elevation/run_east_river_distributed_workflow_light.py'
config = repo_root / 'ess-project/0_config_files/config_East_River_distributed_seasonal_bigBuckt.yaml'

print(f'Repo root: {repo_root}')
print(f'Runner: {runner}')
print(f'Config: {config}')
print(f'Runner exists: {runner.exists()}')
print(f'Config exists: {config.exists()}')

## Configure Run Options

- `preset` can be `minimal`, `light`, or `full`
- Set `with_model_run=True` to include `run_models`
- Set `with_parallel_summa=True` to request local MPI SUMMA when `with_model_run=True`

In [ ]:
preset = 'minimal'
with_model_run = False
with_parallel_summa = False
reuse_domain = ''
steps_override = ''

assert preset in {'minimal', 'light', 'full'}
if with_parallel_summa and not with_model_run:
    print('Note: with_parallel_summa is ignored unless with_model_run=True')

In [ ]:
cmd = [
    'python',
    str(runner),
    '--config', str(config),
    '--preset', preset,
]

if steps_override.strip():
    cmd.extend(['--steps', steps_override.strip()])
if reuse_domain.strip():
    cmd.extend(['--reuse-domain', reuse_domain.strip()])
if with_model_run:
    cmd.append('--with-model-run')
if with_parallel_summa:
    cmd.append('--with-parallel-summa')

print('Command:')
print(' '.join(shlex.quote(part) for part in cmd))

In [ ]:
result = subprocess.run(
    cmd,
    cwd=repo_root,
    text=True,
    capture_output=True,
)

print('Exit code:', result.returncode)
print('--- STDOUT ---')
print(result.stdout)
print('--- STDERR ---')
print(result.stderr)

if result.returncode != 0:
    raise RuntimeError('Lightweight distributed runner failed')

## Quick Preset Guide

- `minimal`: fastest smoke test (`setup_project`, `discretize_domain`, `preprocess_models`)
- `light`: includes observed data and model-agnostic preprocessing
- `full`: includes `run_models` (also enabled automatically by `with_model_run=True`)